# ReServe: Kaggle launcher

Run reusable code from the repository. This notebook first performs a **CPU mock setup check**, not an LLM experiment. The supplied harness does not implement live precision switching or real KV-cache management.

Put the extracted repository at `/kaggle/working/ReServe` before running. For private GitHub repositories, use a private Kaggle input ZIP or your own secure authentication workflow; never embed tokens in clone URLs or cells. See `docs/kaggle.md`.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# Change this if you copied the repository to a different location.
REPO_DIR = Path("/kaggle/working/ReServe")
if not REPO_DIR.exists():
    candidate = Path.cwd()
    if candidate.name == "notebooks":
        candidate = candidate.parent
    if (candidate / "pyproject.toml").exists():
        REPO_DIR = candidate
    else:
        raise FileNotFoundError("Copy/extract the repository first; see docs/kaggle.md")
os.chdir(REPO_DIR)
print("Repository:", REPO_DIR)

In [ ]:
# Base package and plotting only. No automatic replacement of Kaggle's GPU stack.
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[analysis]"], check=True)
subprocess.run([sys.executable, "scripts/check_environment.py"], check=True)

In [ ]:
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], check=True)

In [ ]:
from datetime import datetime, timezone

run_name = datetime.now(timezone.utc).strftime("smoke-%Y%m%dT%H%M%S-%fZ")
RUN_DIR = REPO_DIR / "results" / run_name
subprocess.run(
    [
        sys.executable,
        "-m",
        "recovery_serving.run",
        "--config",
        "configs/smoke.yaml",
        "--output",
        str(RUN_DIR),
    ],
    check=True,
)
subprocess.run([sys.executable, "scripts/plot_run.py", str(RUN_DIR)], check=True)

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(RUN_DIR / "timeline.png")))
print((RUN_DIR / "summary.json").read_text())

## Before real inference

Choose a model and GPU-compatible HF or vLLM stack. See `docs/architecture.md` before editing the scheduler. A streaming client handles baseline requests; token-level scheduling, KV placement, and state-preserving weight switching require engine integration.

After validating a GPU setup, save dependency versions beside its outputs. Save/download each valuable run before ending your Kaggle session.


In [ ]:
# Store environment metadata and create a portable run archive.
import shutil

with (RUN_DIR / "requirements.txt").open("w") as handle:
    subprocess.run([sys.executable, "-m", "pip", "freeze"], stdout=handle, check=True)
archive = shutil.make_archive(str(RUN_DIR), "zip", root_dir=RUN_DIR)
print("Save this output before ending the session:", archive)